## 1. Setup and Data Loading
Imports libraries and loads the protein dataset. Sequences are cleaned.

In [ ]:
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from torch.nn.utils.rnn import pad_sequence
from tqdm import tqdm
import torch.nn as nn
import numpy as np
import math

# Load the dataset
df = pd.read_csv('/home/users/ntu/ktang022/scratch/SC4001_Assignment2/data/2018-06-06-pdb-intersect-pisces.csv')

# Ensure there's a 'len' column with sequence lengths
if 'len' not in df.columns:
    df['len'] = df['seq'].str.len()

# Pre-process sequences
df['seq'] = df['seq'].str.replace("*", "X") # Replace non-standard aa
df = df[df['has_nonstd_aa'] == False].reset_index(drop=True)

print(df.head())
df.info()

## 2. Create Vocabularies
We create a vocabulary for the input amino acid sequences (`seq_vocab`) in addition to the label vocabularies. No ESM-2 model is loaded.

In [ ]:
# Vocabularies for SST8 and SST3 labels
ss8_vocab = {'H': 0, 'G': 1, 'I': 2, 'E': 3, 'B': 4, 'T': 5, 'S': 6, 'C': 7}
ss3_vocab = {'H': 0, 'E': 1, 'C': 2}

# Create vocabulary for input amino acid sequences
all_chars = set(''.join(df['seq']))
seq_vocab = {char: i+1 for i, char in enumerate(sorted(list(all_chars)))}
seq_vocab['<pad>'] = 0 # Add padding token
vocab_size = len(seq_vocab)

print(f"Sequence vocab size: {vocab_size}")
print(seq_vocab)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## 3. Define Dataset and Collate Function
This dataset returns tokenized sequences, not embeddings. The collate function pads batches of sequences and labels.

In [ ]:
class ProteinSequenceDataset(Dataset):
    def __init__(self, sequences, sst8_labels, sst3_labels, seq_vocab, ss8_vocab, ss3_vocab):
        self.sequences = sequences
        self.sst8_labels = sst8_labels
        self.sst3_labels = sst3_labels
        self.seq_vocab = seq_vocab
        self.ss8_vocab = ss8_vocab
        self.ss3_vocab = ss3_vocab

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        seq = self.sequences[idx]
        ss8 = self.sst8_labels[idx]
        ss3 = self.sst3_labels[idx]
        
        # Tokenize sequence
        seq_tokens = [self.seq_vocab.get(c, 0) for c in seq] 
        
        # Tokenize labels
        ss8_tokens = [self.ss8_vocab.get(c, -1) for c in ss8]
        ss3_tokens = [self.ss3_vocab.get(c, -1) for c in ss3]
        
        # Ensure label length matches sequence length
        ss8_tokens = ss8_tokens[:len(seq_tokens)]
        ss3_tokens = ss3_tokens[:len(seq_tokens)]
        
        return torch.tensor(seq_tokens, dtype=torch.long), torch.tensor(ss8_tokens, dtype=torch.long), torch.tensor(ss3_tokens, dtype=torch.long)

def collate_fn(batch):
    seqs, ss8s, ss3s = zip(*batch)
    
    # Pad sequences
    padded_seqs = pad_sequence(seqs, batch_first=True, padding_value=seq_vocab['<pad>'])
    
    # Pad labels (use -1 for padding)
    padded_ss8s = pad_sequence(ss8s, batch_first=True, padding_value=-1)
    padded_ss3s = pad_sequence(ss3s, batch_first=True, padding_value=-1)
    
    return padded_seqs, padded_ss8s, padded_ss3s

## 4. Split Data and Create Dataloaders
The `DataLoader` now uses our custom `collate_fn`.

In [ ]:
# Split indices
train_indices, temp_indices = train_test_split(range(len(df)), test_size=0.2, random_state=42)
val_indices, test_indices = train_test_split(temp_indices, test_size=0.5, random_state=42)

# Create datasets
train_dataset = ProteinSequenceDataset(
    df.iloc[train_indices]['seq'].tolist(),
    df.iloc[train_indices]['sst8'].tolist(),
    df.iloc[train_indices]['sst3'].tolist(),
    seq_vocab, ss8_vocab, ss3_vocab
)
val_dataset = ProteinSequenceDataset(
    df.iloc[val_indices]['seq'].tolist(),
    df.iloc[val_indices]['sst8'].tolist(),
    df.iloc[val_indices]['sst3'].tolist(),
    seq_vocab, ss8_vocab, ss3_vocab
)
test_dataset = ProteinSequenceDataset(
    df.iloc[test_indices]['seq'].tolist(),
    df.iloc[test_indices]['sst8'].tolist(),
    df.iloc[test_indices]['sst3'].tolist(),
    seq_vocab, ss8_vocab, ss3_vocab
)

# Create dataloaders with the collate_fn
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False, collate_fn=collate_fn)

# Define embedding dim as a hyperparameter
embedding_dim = 128

## 5. Define the CNN-Transformer Model (with Learned Embeddings)
This hybrid model includes:
- `nn.Embedding` layer for converting token IDs to embeddings (embedding_dim = 128)
- CNN layers for local feature extraction (kernels 3 & 5, filters = 128)
- `PositionalEncoding` for position information (applied after CNN)
- Transformer encoder layers for capturing long-range dependencies (4 layers, 8 heads)
- Two classification heads for Q8 and Q3 predictions

**Architecture Flow**: Token IDs → Embedding → CNN (local patterns) → Positional Encoding → Transformer (global dependencies) → Classification Heads

In [ ]:
class PositionalEncoding(nn.Module):
    """Standard Transformer Positional Encoding"""
    def __init__(self, d_model, dropout=0.1, max_len=5000):
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(p=dropout)

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)

class CNNTransformer(nn.Module):
    def __init__(self, vocab_size, embedding_dim=128, num_filters=128, 
                 num_heads=8, num_layers=4, dim_feedforward=512, dropout=0.1):
        super().__init__()
        
        # Embedding layer
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=seq_vocab['<pad>'])
        
        # CNN layers for local feature extraction (applied before positional encoding)
        self.conv1 = nn.Conv1d(in_channels=embedding_dim, out_channels=num_filters, kernel_size=3, padding=1)
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(dropout)
        
        self.conv2 = nn.Conv1d(in_channels=num_filters, out_channels=num_filters, kernel_size=5, padding=2)
        self.relu2 = nn.ReLU()
        self.dropout2 = nn.Dropout(dropout)
        
        # Projection layer to match transformer dimension if needed
        self.projection = nn.Linear(num_filters, embedding_dim) if num_filters != embedding_dim else nn.Identity()
        
        # Positional Encoding (applied after CNN)
        self.pos_encoder = PositionalEncoding(embedding_dim, dropout)
        
        # Transformer Encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embedding_dim,
            nhead=num_heads,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        
        # Two separate classifier heads
        self.q8_head = nn.Linear(embedding_dim, 8)
        self.q3_head = nn.Linear(embedding_dim, 3)

    def forward(self, x, mask=None):
        """
        x: [batch_size, seq_len] (token IDs)
        mask: [batch_size, seq_len] (optional, True for valid positions)
        """
        # 1. Convert token IDs to embeddings
        x = self.embedding(x)  # [batch, seq_len, embedding_dim]
        
        # 2. CNN feature extraction (before positional encoding)
        # Permute to [batch, channels, seq_len] for Conv1d
        x = x.permute(0, 2, 1)
        x = self.dropout1(self.relu1(self.conv1(x)))
        x = self.dropout2(self.relu2(self.conv2(x)))
        # Permute back to [batch, seq_len, channels]
        x = x.permute(0, 2, 1)
        
        # 3. Project to transformer dimension
        x = self.projection(x)  # [batch, seq_len, embedding_dim]
        
        # 4. Add positional encoding
        x = self.pos_encoder(x)
        
        # 5. Create padding mask for transformer (True = padding, should be masked)
        if mask is None:
            # Create mask from input (assuming padding token is 0)
            src_key_padding_mask = (x.sum(dim=-1) == 0)  # [batch, seq_len]
        else:
            src_key_padding_mask = ~mask
        
        # 6. Transformer encoding
        x = self.transformer_encoder(x, src_key_padding_mask=src_key_padding_mask)
        
        # 7. Per-residue classification
        q8_logits = self.q8_head(x)
        q3_logits = self.q3_head(x)
        
        return q8_logits, q3_logits

### Model Architecture Summary

| Layer | Operation | Output Dimension | Activation | Dropout |
|-------|-----------|------------------|------------|---------|
| 0 | Embedding | 128 | – | – |
| 1 | Conv1D (kernel=3, filters=128) | 128 | ReLU | 0.1 |
| 2 | Conv1D (kernel=5, filters=128) | 128 | ReLU | 0.1 |
| 3 | Linear Projection | 128 | – | – |
| 4 | Positional Encoding | 128 | – | 0.1 |
| 5 | Transformer Encoder (4 layers, 8 heads, FFN=512) | 128 | – | 0.1 |
| 6 | Q8 Classification Head | 8 | Softmax | – |
| 7 | Q3 Classification Head | 3 | Softmax | – |

## 6. K-Fold Cross-Validation with Early Stopping
Trains the CNN-Transformer model using 5-fold cross-validation with early stopping for robust evaluation.

### Training Configuration

**Training Strategy:**
- 🔄 **5-Fold Cross-Validation** for robust evaluation
- 🛑 **Early Stopping** (patience=7, min_delta=0.0001)
- 📊 **50 Epochs** maximum per fold
- 💾 Saves best model per fold based on validation Q8 accuracy

**Hyperparameters:**
- Batch size: 16
- Learning rate: 1e-4
- Optimizer: Adam
- Loss: CrossEntropyLoss (Q8 + 0.5*Q3)

**Expected Training Time:** ~10-15 hours on GPU (depends on early stopping)

In [ ]:
def compute_accuracy(pred_logits, labels):
    """Per-residue accuracy ignoring -1 padding"""
    preds = pred_logits.argmax(-1)
    mask = labels != -1
    correct = (preds[mask] == labels[mask]).sum().item()
    total = mask.sum().item()
    return correct / total if total > 0 else 0.0

# --- SOV Score Functions ---
q3_id_to_char = {0: 'H', 1: 'E', 2: 'C'}

def get_segments(sequence_chars, state):
    segments = []
    start = -1
    for i, char in enumerate(sequence_chars):
        if char == state:
            if start == -1:
                start = i
        elif start != -1:
            segments.append((start, i - 1))
            start = -1
    if start != -1:
        segments.append((start, len(sequence_chars) - 1))
    return segments

def compute_sov_q3(pred_logits, labels):
    preds = pred_logits.argmax(-1)
    batch_size = preds.shape[0]
    batch_sov_score = 0.0
    
    for i in range(batch_size):
        pred_seq = preds[i]
        true_seq = labels[i]
        mask = true_seq != -1
        pred_seq_filtered = pred_seq[mask]
        true_seq_filtered = true_seq[mask]
        if len(true_seq_filtered) == 0:
            continue
        pred_chars = [q3_id_to_char.get(pid.item(), 'C') for pid in pred_seq_filtered]
        true_chars = [q3_id_to_char.get(tid.item(), 'C') for tid in true_seq_filtered]

        total_weighted_sov = 0.0
        total_residues = 0.0
        for state in ['H', 'E', 'C']:
            true_segments = get_segments(true_chars, state)
            pred_segments = get_segments(pred_chars, state)
            state_residues = sum(1 for char in true_chars if char == state)
            total_residues += state_residues
            if not true_segments:
                continue
            for obs_start, obs_end in true_segments:
                len_obs = (obs_end - obs_start + 1)
                best_min_ov, best_max_ov, best_len_pred = 0, len_obs, 0
                for pred_start, pred_end in pred_segments:
                    overlap_start = max(obs_start, pred_start)
                    overlap_end = min(obs_end, pred_end)
                    min_ov = max(0, overlap_end - overlap_start + 1)
                    if min_ov > 0:
                        max_ov = max(obs_end, pred_end) - min(obs_start, pred_start) + 1
                        len_pred = (pred_end - pred_start + 1)
                        if min_ov > best_min_ov:
                            best_min_ov = min_ov
                            best_max_ov = max_ov
                            best_len_pred = len_pred
                if best_min_ov > 0:
                    delta = min(best_max_ov - best_min_ov, best_min_ov, len_obs // 2, best_len_pred // 2)
                    segment_sov = (best_min_ov + delta) / best_max_ov
                else:
                    segment_sov = 0.0
                total_weighted_sov += (segment_sov * len_obs)
        if total_residues > 0:
            batch_sov_score += (total_weighted_sov / total_residues)
    return batch_sov_score / batch_size
# --- End SOV Functions ---

from sklearn.model_selection import KFold

# Training hyperparameters
num_epochs = 50
patience = 7  # Early stopping patience
min_delta = 0.0001  # Minimum improvement threshold
k_folds = 5
batch_size = 16
learning_rate = 1e-4

# Losses
criterion_q8 = nn.CrossEntropyLoss(ignore_index=-1)
criterion_q3 = nn.CrossEntropyLoss(ignore_index=-1)

# K-Fold Cross-Validation
kfold = KFold(n_splits=k_folds, shuffle=True, random_state=42)

# Store results for each fold
fold_results = []

# Combine train and val for k-fold (keep test separate)
train_val_indices = train_indices + val_indices

print(f"Starting {k_folds}-Fold Cross-Validation with Early Stopping")
print(f"Total epochs: {num_epochs}, Patience: {patience}\n")

for fold, (train_idx, val_idx) in enumerate(kfold.split(train_val_indices)):
    print(f"\n{'='*60}")
    print(f"FOLD {fold + 1}/{k_folds}")
    print(f"{'='*60}")
    
    # Get actual indices for this fold
    fold_train_indices = [train_val_indices[i] for i in train_idx]
    fold_val_indices = [train_val_indices[i] for i in val_idx]
    
    # Create datasets for this fold
    fold_train_dataset = ProteinSequenceDataset(
        df.iloc[fold_train_indices]['seq'].tolist(),
        df.iloc[fold_train_indices]['sst8'].tolist(),
        df.iloc[fold_train_indices]['sst3'].tolist(),
        seq_vocab, ss8_vocab, ss3_vocab
    )
    fold_val_dataset = ProteinSequenceDataset(
        df.iloc[fold_val_indices]['seq'].tolist(),
        df.iloc[fold_val_indices]['sst8'].tolist(),
        df.iloc[fold_val_indices]['sst3'].tolist(),
        seq_vocab, ss8_vocab, ss3_vocab
    )
    
    # Create dataloaders for this fold
    fold_train_loader = DataLoader(fold_train_dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
    fold_val_loader = DataLoader(fold_val_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)
    
    # Initialize model for this fold
    model = CNNTransformer(
        vocab_size=vocab_size, 
        embedding_dim=embedding_dim,
        num_filters=128,
        num_heads=8,
        num_layers=4,
        dim_feedforward=512,
        dropout=0.1
    )
    
    if torch.cuda.device_count() > 1:
        model = nn.DataParallel(model)
    model.to(device)
    
    # Optimizer for this fold
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    
    # Early stopping variables
    best_val_acc_q8 = 0.0
    epochs_no_improve = 0
    early_stop = False
    
    for epoch in range(num_epochs):
        if early_stop:
            print(f"Early stopping triggered at epoch {epoch}")
            break
        model.train()
        train_loss, train_acc_q8, train_acc_q3, train_sov_q3 = 0, 0, 0, 0
        
        for seqs, ss8, ss3 in tqdm(fold_train_loader, desc=f"Fold {fold+1}, Epoch {epoch+1}/{num_epochs}"):
            seqs, ss8, ss3 = seqs.to(device), ss8.to(device), ss3.to(device)
            
            # Forward pass
            q8_logits, q3_logits = model(seqs)
            
            # Loss
            loss_q8 = criterion_q8(q8_logits.view(-1, 8), ss8.view(-1))
            loss_q3 = criterion_q3(q3_logits.view(-1, 3), ss3.view(-1))
            loss = loss_q8 + 0.5 * loss_q3
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
            train_acc_q8 += compute_accuracy(q8_logits, ss8)
            train_acc_q3 += compute_accuracy(q3_logits, ss3)
            train_sov_q3 += compute_sov_q3(q3_logits, ss3)
        
        train_loss /= len(fold_train_loader)
        train_acc_q8 /= len(fold_train_loader)
        train_acc_q3 /= len(fold_train_loader)
        train_sov_q3 /= len(fold_train_loader)
        
        # Validation
        model.eval()
        val_loss, val_acc_q8, val_acc_q3, val_sov_q3 = 0, 0, 0, 0
        with torch.no_grad():
            for seqs, ss8, ss3 in fold_val_loader:
                seqs, ss8, ss3 = seqs.to(device), ss8.to(device), ss3.to(device)
                q8_logits, q3_logits = model(seqs)
                
                loss_q8 = criterion_q8(q8_logits.view(-1, 8), ss8.view(-1))
                loss_q3 = criterion_q3(q3_logits.view(-1, 3), ss3.view(-1))
                loss = loss_q8 + 0.5 * loss_q3
                
                val_loss += loss.item()
                val_acc_q8 += compute_accuracy(q8_logits, ss8)
                val_acc_q3 += compute_accuracy(q3_logits, ss3)
                val_sov_q3 += compute_sov_q3(q3_logits, ss3)
        
        val_loss /= len(fold_val_loader)
        val_acc_q8 /= len(fold_val_loader)
        val_acc_q3 /= len(fold_val_loader)
        val_sov_q3 /= len(fold_val_loader)
        
        print(f"Epoch {epoch+1}: Train Loss={train_loss:.4f}, Val Loss={val_loss:.4f}")
        print(f"Train Q8={train_acc_q8:.4f}, Val Q8={val_acc_q8:.4f}, Train Q3={train_acc_q3:.4f}, Val Q3={val_acc_q3:.4f}, Val SOV={val_sov_q3:.4f}")
        
        # Early stopping check
        if val_acc_q8 > best_val_acc_q8 + min_delta:
            best_val_acc_q8 = val_acc_q8
            epochs_no_improve = 0
            # Save best model for this fold
            model_state = model.module.state_dict() if isinstance(model, nn.DataParallel) else model.state_dict()
            torch.save(model_state, f"best_cnn_transformer_fold{fold+1}.pt")
            print(f"✓ New best Q8: {best_val_acc_q8:.4f} - Model saved")
        else:
            epochs_no_improve += 1
            print(f"✗ No improvement for {epochs_no_improve} epoch(s)")
            
        if epochs_no_improve >= patience:
            print(f"\n⚠ Early stopping triggered! No improvement for {patience} epochs.")
            early_stop = True
    
    # Store fold results
    fold_results.append({
        'fold': fold + 1,
        'best_val_q8': best_val_acc_q8,
        'best_val_q3': val_acc_q3,
        'best_val_sov': val_sov_q3,
        'epochs_trained': epoch + 1
    })
    
    print(f"\nFold {fold+1} Summary:")
    print(f"Best Val Q8 Acc: {best_val_acc_q8:.4f}")
    print(f"Epochs trained: {epoch + 1}")

# Print overall K-Fold results
print(f"\n{'='*60}")
print("K-FOLD CROSS-VALIDATION RESULTS")
print(f"{'='*60}")
for result in fold_results:
    print(f"Fold {result['fold']}: Q8={result['best_val_q8']:.4f}, Q3={result['best_val_q3']:.4f}, SOV={result['best_val_sov']:.4f}, Epochs={result['epochs_trained']}")

avg_q8 = np.mean([r['best_val_q8'] for r in fold_results])
std_q8 = np.std([r['best_val_q8'] for r in fold_results])
avg_q3 = np.mean([r['best_val_q3'] for r in fold_results])
std_q3 = np.std([r['best_val_q3'] for r in fold_results])
avg_sov = np.mean([r['best_val_sov'] for r in fold_results])
std_sov = np.std([r['best_val_sov'] for r in fold_results])

print(f"\nAverage Q8 Accuracy: {avg_q8:.4f} ± {std_q8:.4f}")
print(f"Average Q3 Accuracy: {avg_q3:.4f} ± {std_q3:.4f}")
print(f"Average SOV Score: {avg_sov:.4f} ± {std_sov:.4f}")

## 7. Final Evaluation on Test Set
Evaluates each fold's best model on the test set and computes ensemble predictions.

## 8. (Optional) Ensemble Prediction
Combines predictions from all 5 fold models for potentially better performance.

In [ ]:
print(f"\n{'='*60}")
print("ENSEMBLE PREDICTION (VOTING)")
print(f"{'='*60}\n")

# Load all fold models
models = []
for fold in range(k_folds):
    model = CNNTransformer(
        vocab_size=vocab_size, 
        embedding_dim=embedding_dim,
        num_filters=128,
        num_heads=8,
        num_layers=4,
        dim_feedforward=512,
        dropout=0.1
    )
    model.load_state_dict(torch.load(f"best_cnn_transformer_fold{fold+1}.pt"))
    if torch.cuda.device_count() > 1:
        model = nn.DataParallel(model)
    model.to(device)
    model.eval()
    models.append(model)

# Ensemble predictions
ensemble_acc_q8, ensemble_acc_q3, ensemble_sov_q3 = 0, 0, 0

with torch.no_grad():
    for seqs, ss8, ss3 in tqdm(test_loader, desc="Ensemble Testing"):
        seqs, ss8, ss3 = seqs.to(device), ss8.to(device), ss3.to(device)
        
        # Collect predictions from all models
        q8_probs_list = []
        q3_probs_list = []
        
        for model in models:
            q8_logits, q3_logits = model(seqs)
            q8_probs_list.append(torch.softmax(q8_logits, dim=-1))
            q3_probs_list.append(torch.softmax(q3_logits, dim=-1))
        
        # Average probabilities (soft voting)
        q8_probs_avg = torch.stack(q8_probs_list).mean(dim=0)
        q3_probs_avg = torch.stack(q3_probs_list).mean(dim=0)
        
        # Convert back to logits for metric computation
        ensemble_q8_logits = torch.log(q8_probs_avg + 1e-10)
        ensemble_q3_logits = torch.log(q3_probs_avg + 1e-10)
        
        ensemble_acc_q8 += compute_accuracy(ensemble_q8_logits, ss8)
        ensemble_acc_q3 += compute_accuracy(ensemble_q3_logits, ss3)
        ensemble_sov_q3 += compute_sov_q3(ensemble_q3_logits, ss3)

ensemble_acc_q8 /= len(test_loader)
ensemble_acc_q3 /= len(test_loader)
ensemble_sov_q3 /= len(test_loader)

print(f"\nEnsemble Test Results:")
print(f"Ensemble Q8 Accuracy: {ensemble_acc_q8:.4f}")
print(f"Ensemble Q3 Accuracy: {ensemble_acc_q3:.4f}")
print(f"Ensemble SOV Score: {ensemble_sov_q3:.4f}")
print(f"\nComparison:")
print(f"Single best fold Q8: {best_fold['test_q8']:.4f}")
print(f"Ensemble Q8: {ensemble_acc_q8:.4f}")
print(f"Improvement: {(ensemble_acc_q8 - best_fold['test_q8']):.4f}")

In [ ]:
# Test each fold's best model
test_results = []

print(f"\n{'='*60}")
print("TESTING INDIVIDUAL FOLD MODELS")
print(f"{'='*60}\n")

for fold in range(k_folds):
    print(f"Testing Fold {fold+1} model...")
    
    # Initialize model
    model = CNNTransformer(
        vocab_size=vocab_size, 
        embedding_dim=embedding_dim,
        num_filters=128,
        num_heads=8,
        num_layers=4,
        dim_feedforward=512,
        dropout=0.1
    )
    
    # Load the best model for this fold
    model.load_state_dict(torch.load(f"best_cnn_transformer_fold{fold+1}.pt"))
    
    if torch.cuda.device_count() > 1:
        model = nn.DataParallel(model)
    model.to(device)
    model.eval()
    
    test_loss, test_acc_q8, test_acc_q3, test_sov_q3 = 0, 0, 0, 0
    with torch.no_grad():
        for seqs, ss8, ss3 in test_loader:
            seqs, ss8, ss3 = seqs.to(device), ss8.to(device), ss3.to(device)
            q8_logits, q3_logits = model(seqs)
            
            loss_q8 = criterion_q8(q8_logits.view(-1, 8), ss8.view(-1))
            loss_q3 = criterion_q3(q3_logits.view(-1, 3), ss3.view(-1))
            loss = loss_q8 + 0.5 * loss_q3
            
            test_loss += loss.item()
            test_acc_q8 += compute_accuracy(q8_logits, ss8)
            test_acc_q3 += compute_accuracy(q3_logits, ss3)
            test_sov_q3 += compute_sov_q3(q3_logits, ss3)
    
    test_loss /= len(test_loader)
    test_acc_q8 /= len(test_loader)
    test_acc_q3 /= len(test_loader)
    test_sov_q3 /= len(test_loader)
    
    test_results.append({
        'fold': fold + 1,
        'test_q8': test_acc_q8,
        'test_q3': test_acc_q3,
        'test_sov': test_sov_q3,
        'test_loss': test_loss
    })
    
    print(f"Fold {fold+1} Test - Q8: {test_acc_q8:.4f}, Q3: {test_acc_q3:.4f}, SOV: {test_sov_q3:.4f}\n")

# Summary statistics
print(f"{'='*60}")
print("FINAL TEST SET RESULTS (K-FOLD AVERAGE)")
print(f"{'='*60}")

avg_test_q8 = np.mean([r['test_q8'] for r in test_results])
std_test_q8 = np.std([r['test_q8'] for r in test_results])
avg_test_q3 = np.mean([r['test_q3'] for r in test_results])
std_test_q3 = np.std([r['test_q3'] for r in test_results])
avg_test_sov = np.mean([r['test_sov'] for r in test_results])
std_test_sov = np.std([r['test_sov'] for r in test_results])

print(f"\nTest Q8 Accuracy: {avg_test_q8:.4f} ± {std_test_q8:.4f}")
print(f"Test Q3 Accuracy: {avg_test_q3:.4f} ± {std_test_q3:.4f}")
print(f"Test SOV Score: {avg_test_sov:.4f} ± {std_test_sov:.4f}")

# Find best performing fold
best_fold = max(test_results, key=lambda x: x['test_q8'])
print(f"\nBest performing fold: Fold {best_fold['fold']} (Q8: {best_fold['test_q8']:.4f})")